Archived notebook excerpt: manuscript 100-gene branch. Outputs were removed. These cells preserve the recorded logic, including diagnostics; use the portable scripts in this directory for fresh execution. This is not evidence that training completed. Original source cell indices: 0, 8, 14, 15, 16, 17, 18, 19, 20, 21, 29, 30, 31.

In [ ]:
# 导入必要的库
import sys
import os
import pandas as pd  # 数据处理
import scanpy as sc  # 单细胞 RNA-seq 分析
import numpy as np   # 数值计算
import matplotlib.pyplot as plt  # 数据可视化
from matplotlib.pyplot import rc_context
import anndata  # AnnData 数据结构
import seaborn as sns  # 高级数据可视化
import torch  # 深度学习框架
import matplotlib.font_manager
from matplotlib import rcParams

# 设置全局字体和图形参数
font_list = []
fpaths = matplotlib.font_manager.findSystemFonts()
for i in fpaths:
    try:
        f = matplotlib.font_manager.get_font(i)
        font_list.append(f.family_name)
    except RuntimeError:
        pass

font_list = set(font_list)
plot_font = 'Helvetica' if 'Helvetica' in font_list else 'FreeSans'
rcParams['font.family'] = plot_font  # 设置字体
rcParams.update({'font.size': 10})  # 字体大小
rcParams.update({'figure.dpi': 300})  # 图像分辨率
rcParams.update({'figure.figsize': (3, 3)})  # 图形默认大小
rcParams.update({'savefig.dpi': 500})  # 保存图像分辨率

# 自动重载模块，便于调试
%load_ext autoreload
%autoreload 2

# 添加项目路径，便于导入自定义模块
sys.path.append('../')
import sample_squidiff  # 导入自定义模块

# 忽略运行时警告
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 初始化采样器，加载预训练模型
sampler = sample_squidiff.sampler(
    model_path = rf"mouse_targeted_100/model.pt",  # 模型路径
    gene_size = 100,  # 输入基因数量
    output_dim = 100,  # 输出基因数量
    use_drug_structure = False  # 不使用药物分子结构
)

In [ ]:
# === CELL 1: 读取数据 ===
import torch
import scanpy as sc
import numpy as np
import os
import gc
from Squidiff.script_util import model_and_diffusion_defaults, create_model_and_diffusion

# 1. 垃圾回收，确保显存干净
gc.collect()
torch.cuda.empty_cache()

# 2. 设定路径
DATA_PATH = 'Radiation-data/predict_targeted.h5ad' 
MODEL_PATH = 'logs/mouse_targeted_genes/model.pt'
GENE_SIZE = 100

print(f"🔹 正在读取文件: {DATA_PATH}")
adata_final = sc.read_h5ad(DATA_PATH)

# 3.以此处读到的行数为准，作为后续所有操作的标准
TARGET_COUNT = adata_final.shape[0]

print(f"✅ 数据读取成功！")
print(f"   - 细胞总数 (TARGET_COUNT): {TARGET_COUNT}")
print(f"   - 基因总数: {adata_final.shape[1]}")

# 🛑 检查点：如果这里显示的数字是 2700，那说明文件本身就被覆盖了，不用往下跑了。
if TARGET_COUNT < 10000:
    raise ValueError(f"❌ 严重警告：读入的文件只有 {TARGET_COUNT} 行！请检查是否覆盖了原文件！")
else:
    print("🎉 第一步检查通过：文件大小正常。")

In [ ]:
# === CELL 2: 转换 Tensor ===
print("🔹 正在将数据转换为 Tensor...")

# 1. 处理稀疏矩阵 (防止格式报错)
if hasattr(adata_final.X, 'toarray'):
    X_dense = adata_final.X.toarray()
else:
    X_dense = adata_final.X

# 2. 搬运到 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tensor_final = torch.tensor(X_dense, dtype=torch.float32).to(device)

print(f"✅ Tensor 创建完成")
print(f"   - Tensor 形状: {tensor_final.shape}")

# 🛑 检查点：再次确认形状
if tensor_final.shape[0] != TARGET_COUNT:
    raise RuntimeError(f"❌ 第二步失败：Tensor 行数 ({tensor_final.shape[0]}) 与数据行数不一致！")
else:
    print("🎉 第二步检查通过：Tensor 准备就绪。")

In [ ]:
# === CELL 3: 加载模型 (最终修正路径版) ===
import os
import torch
from Squidiff.script_util import model_and_diffusion_defaults, create_model_and_diffusion

print("🔹 正在加载模型权重...")

# 1. 【修正】去掉 logs/，直接指向文件夹
MODEL_PATH = 'mouse_targeted_100/model.pt' 

# 再次自动检查一下，防止文件名不是 model.pt (比如是 model050000.pt)
if not os.path.exists(MODEL_PATH):
    dir_name = 'mouse_targeted_100'
    print(f"❌ 还是没找到文件: {MODEL_PATH}")
    if os.path.exists(dir_name):
        print(f"   但是 '{dir_name}' 文件夹是存在的！里面的文件有：")
        files = os.listdir(dir_name)
        print(f"   {files}")
        print("   👉 如果里面是 model010000.pt 这种名字，请手动修改代码里的 MODEL_PATH！")
    else:
        print(f"   连 '{dir_name}' 这个文件夹都没找到，请检查左边文件列表。")
    raise FileNotFoundError("路径不对，请根据提示修改！")

# 2. 参数设置
args_dict = model_and_diffusion_defaults()
args_dict.update({
    'gene_size': GENE_SIZE,
    'output_dim': GENE_SIZE,
    'use_drug_structure': False,
    'num_channels': 128,
    'use_checkpoint': False,
    'class_cond': False,
    'use_scale_shift_norm': True,
    'use_encoder': True  # 关键参数
})

try:
    # 3. 搭建骨架
    model, diffusion = create_model_and_diffusion(**args_dict)
    
    # 4. 加载权重
    # map_location 确保在 CPU/GPU 都能加载
    state_dict = torch.load(MODEL_PATH, map_location=device)
    model.load_state_dict(state_dict)
    
    model.to(device)
    model.eval() 
    print(f"🎉 第三步完美通过！成功加载模型: {MODEL_PATH}")
    
except Exception as e:
    print(f"❌ 第三步失败 -> {e}")

In [ ]:
# === CELL 4 (智能修复版): 提取向量 ===
print(f"🔹 开始提取向量，目标总数: {TARGET_COUNT}")

batch_size = 64 # 保持这个稳健的批次大小
curr_idx = 0
z_list = []

# 自动探测器：跑第一个 batch 看看输出到底是啥
if curr_idx == 0:
    test_batch = tensor_final[0:batch_size]
    with torch.no_grad():
        test_out = model.encoder(test_batch)
    
    # 🕵️‍♂️ 侦探逻辑
    use_index_0 = False
    if isinstance(test_out, tuple) or isinstance(test_out, list):
        print(f"   [探测] 模型返回的是 Tuple/List，元素数量: {len(test_out)}")
        print(f"   [探测] 第一个元素形状: {test_out[0].shape}")
        use_index_0 = True
    else:
        print(f"   [探测] 模型直接返回 Tensor，形状: {test_out.shape}")
        use_index_0 = False
        
    if use_index_0 and len(test_out[0].shape) == 1:
        print("   ⚠️ 警告：取下标 [0] 后变成了 1维数组，这不对！可能不需要取下标。")

# 开始正式循环
with torch.no_grad():
    while curr_idx < TARGET_COUNT:
        end_idx = min(curr_idx + batch_size, TARGET_COUNT)
        
        # 切片
        batch = tensor_final[curr_idx : end_idx]
        
        # 运行模型
        out = model.encoder(batch)
        
        # 根据探测结果决定怎么取值
        if use_index_0:
            z = out[0]
        else:
            z = out # <--- 关键修改：直接拿结果
            
        z_list.append(z.cpu().numpy())
        
        # 进度
        if curr_idx % 5000 == 0:
             print(f"   >>> 进度: {curr_idx} / {TARGET_COUNT} ...")
        curr_idx = end_idx

# 拼接
z_result = np.concatenate(z_list, axis=0)

print(f"✅ 提取结束。")
print(f"   - 最终形状: {z_result.shape} (预期应该是 {TARGET_COUNT} x 100)")

if len(z_result.shape) == 2 and z_result.shape[1] == 100:
    print("🎉 形状正确！是 2D 矩阵。请继续运行 CELL 5 和 6。")
else:
    print("❌ 形状依然不对，请把上面的 [探测] 日志发给我。")

In [ ]:
# === CELL 5: 核心计算与预测 ===
# 1. 确保数据是 z_result (从上一步继承)
if 'z_result' not in locals():
    raise ValueError("❌ 请先运行 CELL 4 提取向量！")

# 2. 赋值回 AnnData
adata_final.obsm['X_squidiff'] = z_result
print(f"✅ 已保存向量，维度: {z_result.shape}")

# 3. 提取索引
idx_ctrl = adata_final.obs['Group'] == 'Ctrl'
idx_ir = adata_final.obs['Group'] == 'IR'
idx_dmso = adata_final.obs['Group'] == 'DMSO'
idx_dmso_ir = adata_final.obs['Group'] == 'DMSO-IR'

print(f"   各组数量 -> Ctrl:{sum(idx_ctrl)}, IR:{sum(idx_ir)}, DMSO:{sum(idx_dmso)}, DMSO-IR:{sum(idx_dmso_ir)}")

# 4. 核心公式计算
# 计算重心
vec_ctrl = np.mean(z_result[idx_ctrl], axis=0)
vec_ir = np.mean(z_result[idx_ir], axis=0)
vec_dmso_real = np.mean(z_result[idx_dmso_ir], axis=0)  # 真实答案

# 算出辐射向量 (Shift)
delta_radiation = vec_ir - vec_ctrl

# 预测 DMSO + 辐射
z_dmso_cells = z_result[idx_dmso]
z_pred = z_dmso_cells + delta_radiation

# 计算预测组的重心
vec_dmso_ir_pred = np.mean(z_pred, axis=0)

# 5. 打分
# 使用 try-except 捕获潜在的计算错误
try:
    dist_effect = np.linalg.norm(vec_ir - vec_ctrl)  # 辐射本身造成的改变量
    dist_error = np.linalg.norm(vec_dmso_real - vec_dmso_ir_pred)  # 预测偏差
    accuracy = 1 - (dist_error / dist_effect)

    print(f"\n🏆 ======== 最终结果报告 ========")
    print(f"1. 辐射强度 (Effect Size): {dist_effect:.4f}")
    print(f"2. 预测误差 (Error):       {dist_error:.4f}")
    print(f"3. 准确度 (Accuracy):      {accuracy:.4f}")

    if accuracy > 0.5:
        print("🌟 成功！模型学到了有效的辐射向量！")
    else:
        print("⚠️ 准确度较低，请检查数据或模型。")
except Exception as e:
    print(f"❌ 计算过程中出现错误: {e}")

In [ ]:
# === CELL 6: 绘图验证 ===
import matplotlib.pyplot as plt

print("🔹 正在绘图...")

# 1. 准备数据
# 确保 z_result 和 z_pred 的形状一致
if z_result[idx_dmso_ir].shape[1] != z_pred.shape[1]:
    raise ValueError("❌ 数据形状不一致，无法绘图！请检查 z_result 和 z_pred 的维度。")

z_vis = np.vstack([
    z_result[idx_dmso_ir],  # 真实值 (蓝)
    z_pred                  # 预测值 (红)
])

# 2. 打标签
labels = ['Real DMSO-IR'] * sum(idx_dmso_ir) + ['Predicted DMSO-IR'] * len(z_pred)

# 3. 创建临时对象画图
adata_vis = sc.AnnData(z_vis)
adata_vis.obs['Condition'] = labels
sc.tl.pca(adata_vis)

# 4. 绘图
fig, ax = plt.subplots(figsize=(6, 5))
sc.pl.pca(
    adata_vis, 
    color='Condition', 
    palette={'Real DMSO-IR': '#1f77b4', 'Predicted DMSO-IR': '#d62728'}, # 蓝实 vs 红虚
    alpha=0.6, 
    size=30,
    title=f'Accuracy Score: {accuracy:.3f}',
    ax=ax
)
plt.show()

# 5. 保存最终预测结果
# 添加 try-except 捕获潜在的保存错误
try:
    adata_save = adata_final[idx_dmso].copy()
    adata_save.obsm['X_squidiff_pred'] = z_pred
    save_path = "Radiation-data/final_prediction_result.h5ad"
    adata_save.write(save_path)
    print(f"✅ 预测结果已保存至: {save_path}")
except Exception as e:
    print(f"❌ 保存预测结果时出错: {e}")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import torch

# === 1. 准备数据 ===
# 我们回到基因表达空间 (Gene Space)，而不是 Latent Space
# 确保使用 adata_final，其中 .X 是 100 个基因的表达量
print("🔹 正在计算基因变化的差异...")

# 提取 100 个基因的名字
gene_names = adata_final.var_names.tolist()

# 提取四个组的表达量矩阵 (转为 DataFrame 方便计算)
# 注意：如果 X 是稀疏矩阵，用 toarray()；如果是 dense 直接用
if hasattr(adata_final.X, 'toarray'):
    df = pd.DataFrame(adata_final.X.toarray(), columns=gene_names)
else:
    df = pd.DataFrame(adata_final.X, columns=gene_names)

df['Group'] = adata_final.obs['Group'].values

# === 2. 计算每组的平均表达量 (Mean Expression) ===
group_means = df.groupby('Group').mean()

group_means

In [ ]:
# === 3. 核心计算：分批生成虚拟的 DMSO-IR ===
# 分批处理 z_pred，避免内存问题
batch_size = 64
z_pred_gene_space_list = []

print("🔹 分批生成虚拟 DMSO-IR...")
for i in range(0, z_pred.shape[0], batch_size):
    batch = z_pred[i:i + batch_size]
    with torch.no_grad():
        batch_pred = sampler.pred(torch.tensor(batch).to('cuda'), gene_size=adata_final.shape[1])
        z_pred_gene_space_list.append(batch_pred.detach().cpu().numpy())

# 拼接所有批次结果
z_pred_gene_space = np.vstack(z_pred_gene_space_list)
z_pred_gene_space = pd.DataFrame(z_pred_gene_space, columns=gene_names)
z_pred_gene_space['Group'] = 'Virtual DMSO-IR'

# 将虚拟数据合并到原始数据中
virtual_df = pd.concat([df, z_pred_gene_space], ignore_index=True)

virtual_df

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind

# === 1. 提取真实和虚拟 DMSO-IR 数据 ===
# 真实 DMSO-IR 数据
real_dmso_ir = adata_final[adata_final.obs['Group'] == 'DMSO-IR'].X
if hasattr(real_dmso_ir, 'toarray'):  # 如果是稀疏矩阵，转换为稠密矩阵
    real_dmso_ir = real_dmso_ir.toarray()

print(f"Real DMSO-IR shape: {real_dmso_ir.shape}")
print(f"Real DMSO-IR shape: {real_dmso_ir}")

# 虚拟 DMSO-IR 数据
virtual_dmso_ir = z_pred_gene_space.iloc[:, :-1].values  # 去掉最后一列（Group）
print(f"Virtual DMSO-IR shape: {virtual_dmso_ir.shape}")
print(f"Virtual DMSO-IR shape: {virtual_dmso_ir}")

In [ ]:
import scanpy as sc

# === 1. 准备 AnnData 对象 ===
# 将真实和虚拟 DMSO-IR 数据合并到一个 AnnData 对象中
combined_data = np.vstack([real_dmso_ir, virtual_dmso_ir])
group_labels = ['Real DMSO-IR'] * real_dmso_ir.shape[0] + ['Virtual DMSO-IR'] * virtual_dmso_ir.shape[0]

# 确保基因名称正确
gene_names = adata_final.var_names.tolist()  # 从原始数据中提取基因名称

adata_combined = sc.AnnData(combined_data, var={'Gene': gene_names})
adata_combined.var_names = gene_names  # 设置 var_names 为真实基因名称
adata_combined.obs['Group'] = group_labels

# === 2. 差异基因分析 ===
sc.tl.rank_genes_groups(adata_combined, groupby='Group', method='wilcoxon')

# === 3. 提取结果 ===
result = adata_combined.uns['rank_genes_groups']

# === 4. 打印前 10 个差异基因 ===
print("Top 10 differentially expressed genes:")
for gene, score, pval in zip(result['names']['Real DMSO-IR'][:10], 
                             result['scores']['Real DMSO-IR'][:10], 
                             result['pvals']['Real DMSO-IR'][:10]):
    print(f"Gene: {gene}, Score: {score:.4f}, P-value: {pval:.4e}")

# === 5. 可视化差异基因 ===
sc.pl.rank_genes_groups(adata_combined, n_genes=25, sharey=False)

In [ ]:
# 直接比较两组
sc.tl.rank_genes_groups(adata_combined, groupby='Group', groups=['Real DMSO-IR'], reference='Virtual DMSO-IR', method='wilcoxon')

# 可视化
sc.pl.rank_genes_groups(adata_combined, n_genes=25, sharey=False)